TALLER FINAL

Primero instalamos la libreria que nos permitira consultar la Api.

In [1]:
pip install requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Con la librería requests importada, podemos conectarnos a la API y extraer la información necesaria. 

Usamos la variable base_url para almacenar la dirección principal de la web.
Definimos las rutas específicas de consulta mediante las variables de endpoints endpoint_atracciones, endpoint_ciudades, endpoint_departamentos y endpoint_region.
Finalmente, mediante el método requests.get(), combinamos la URL base con cada endpoint para realizar las peticiones y almacenar las respuestas de las tablas consultadas.

In [59]:
import requests
base_url = "https://api-colombia.com"
endpoint_atracciones = "/api/v1/TouristicAttraction"
endpoint_ciudades = "/api/v1/City"
endpoint_departamentos = "/api/v1/Department"
endpoint_region = "/api/v1/Region"
respuesta_api_atracciones = requests.get(base_url + endpoint_atracciones)
respuesta_api_ciudades = requests.get(base_url + endpoint_ciudades)
respuesta_api_departamentos = requests.get(base_url + endpoint_departamentos)
respuesta_api_region = requests.get(base_url + endpoint_region)

Importamos la libreria pandas para transformar las respuestas de las api en DataFrames para trabar en Power BI y solicitamos bajo el codigo status_code para saber si nos arroja el codigo 200  la cual nos indica que la consulta fue satisfactoria.

In [60]:
import pandas as pd 
print(respuesta_api_atracciones.status_code)
print(respuesta_api_ciudades.status_code)
print(respuesta_api_departamentos.status_code)
print(respuesta_api_region.status_code)
atracciones_json = respuesta_api_atracciones.json()
ciudades_json = respuesta_api_ciudades.json()
departamentos_json = respuesta_api_departamentos.json()
region_json = respuesta_api_region.json()

df_atracciones = pd.DataFrame(atracciones_json)
df_ciudades = pd.DataFrame(ciudades_json)
df_departamentos = pd.DataFrame(departamentos_json)
df_region = pd.DataFrame(region_json)

200
200
200
200


··· Transformación y unión de las APIs consultadas

Para realizar el cruce de datos, asignamos un nuevo DataFrame llamado `df_unificado_1`. Este se encargará de unir la información de las dos tablas de acuerdo con una columna en común o llave de relación. 

Adicionalmente, utilizamos el parámetro `suffixes` para indicarle a Pandas qué sufijo o palabra debe añadir al final del nombre de las columnas en caso de que estas se repitan en ambas tablas, evitando así la duplicidad de nombres.


In [61]:
df_unificado_1 = pd.merge(
    df_atracciones,
    df_ciudades,
    left_on="cityId",
    right_on="id",
    suffixes=("_atraccion", "_ciudad"),
)

con la unión anterior, creamos un nuevo dataFrame donde nos va a unir el resultado anterior con df_departamentos con la columna en común departamentoID y id.

In [62]:
df_unificado_2 = pd.merge(
    df_unificado_1,
    df_departamentos,
    left_on="departmentId",
    right_on="id",
    suffixes=("", "_dep"),
)

con la unión anterior, creamos un nuevo dataFrame donde nos va a unir el resultado anterior con df_region con la columna en común regionId y id.

In [63]:
df_unificado_3 = pd.merge(
    df_unificado_2,
    df_region,
    left_on="regionId",
    right_on="id",
    suffixes=("", "_region"),
)

·Selección y filtrado de columnas finales

Una vez consolidadas las fuentes de información en el DataFrame principal, extraemos únicamente las columnas de interés que utilizaremos para nuestro análisis y reporte final.


In [64]:
df_completo = df_unificado_3[
    [
        "name_atraccion",
        "description_atraccion",
        "latitude",
        "longitude",
        "name_ciudad",
        "population",
        "surface",
        "name",
        "name_region",
    ]
].copy()

Transformamos los nombres a español.

In [65]:
df_completo.columns = [
    "Atraccion",
    "Descripcion",
    "Latitud",
    "Longitud",
    "Ciudad",
    "Poblacion",
    "Superficie",
    "Departamento",
    "Region",
]

Con head() traemos información del dataFrame modificado.

In [66]:
df_completo.head()

,Atraccion,Descripcion,Latitud,Longitud,Ciudad,Poblacion,Superficie,Departamento,Region
0,Hacienda napoles,La Hacienda Nápoles es el nombre que tuviera e...,5.887894,-74.6418,Puerto Triunfo,NaN,NaN,Antioquia,Andina
1,Santa fe de antioquia,Santa Fe de Antioquia3​ es un municipio de Col...,6.556389,-75.8275,Santafé de Antioquia,NaN,NaN,Antioquia,Andina
2,La ventana del mundo,La Ventana al mundo es un monumento público ub...,11.03308,-74.831406,Barranquilla,2326292.0,520.0,Atlántico,Caribe
3,Volcan del totumo,El volcán del Totumo es un cono volcánico llen...,10.744472,-75.241403,Santa Catalina,NaN,NaN,Bolívar,Caribe
4,Lago de Tota,El lago de Tota es un cuerpo de agua natural s...,5.55,-72.916667,Aquitania,NaN,NaN,Boyacá,Andina


Agrupamos los datos por ciudad, se calcula la población, superficie y la cantidad de actracciones por ciudad.

In [67]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df_ciudades_ml = (
    df_completo.groupby(["Ciudad", "Departamento", "Region"])
    .agg(
        Poblacion=("Poblacion", "first"),
        Superficie=("Superficie", "first"),
        Total_Atracciones=("Atraccion", "count"),
    )
    .reset_index()
)



Seleccionamos las varibles para el modelo x, con la linea de codigo df_ciudades_ml.fillna(0, inplace=True) buscamos los valores vacios y se cambian automaticamente por cero.

In [70]:
df_ciudades_ml.fillna(0, inplace=True)
X = df_ciudades_ml[["Poblacion", "Superficie"]]

# 3. Escalado de datos (Obligatorio en K-Means para que la población no opaque a la superficie)
scaler = StandardScaler()
X_escalado = scaler.fit_transform(X)


Darle nombres lógicos a los clusters analizando sus promedios, se entrenar el modelo K-Means con 3 grupos (Clusters)

In [ ]:
# Ajustamos n_init explícitamente para evitar advertencias de la librería
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_ciudades_ml["Cluster_ID"] = kmeans.fit_predict(X_escalado)


def etiquetar_cluster(row):
    if row["Cluster_ID"] == 0:
        return "Municipios Rurales / Extensos"
    elif row["Cluster_ID"] == 1:
        return "Grandes Ciudades / Hubs Urbanos"
    else:
        return "Municipios Medios / En Desarrollo"


df_ciudades_ml["Categoria_Cluster"] = df_ciudades_ml.apply(
    etiquetar_cluster, axis=1
)

Unir los resultados del modelo de vuelta a la base de atracciones que se utilizara en el dashboard.

In [75]:
df_trabajo_final = pd.merge(
    df_completo,
    df_ciudades_ml[["Ciudad", "Categoria_Cluster"]],
    on="Ciudad",
    how="left",
)


Descargamos el archivo con extensión xlsx.

In [76]:
df_trabajo_final.to_excel("datos_colombia_dashboard.xlsx", index=False)